# Data Loading and Baseline Model Training

**1. Data Loading**
This code sets up the CIFAR-10 dataset and data loading pipeline for training and evaluation using PyTorch. It applies basic preprocessing to convert images into tensors and normalize them using the standard CIFAR-10 mean and standard deviation values. A DEBUG flag is included to optionally reduce the training dataset size for faster experimentation during development.

The CIFAR-10 training and test datasets are then downloaded and loaded with their respective transformations. DataLoader objects are created to efficiently batch and shuffle the training data, while keeping the test data in a fixed order for evaluation. Additional settings such as multiple worker processes and pinned memory are used to improve data loading performance during training.

In [ ]:
import os
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim

from models import ResNetCIFAR

# Toggle for fast debugging
DEBUG = False

use_pin_memory = torch.cuda.is_available()
num_workers = min(2, os.cpu_count())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Transforms (no augmentation for now)
transform_train = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

# 2. Datasets
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

# Debug subset
if DEBUG:
    trainset = torch.utils.data.Subset(trainset, range(10000))

# 3. DataLoaders
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=128,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
)

# Optional debug print to check classes
classes = trainset.dataset.classes if DEBUG else trainset.classes

Files already downloaded and verified
Files already downloaded and verified


**2. Instantiate the Baseline Model from `models.py`**
This section keeps the baseline architecture centralized in `models.py` and imports `ResNetCIFAR` directly into the notebook. This avoids duplicate class definitions across notebooks and makes future updates easier to maintain.

After importing, the model is moved to the available device (GPU if available), and the total number of trainable parameters is computed and printed to keep the same baseline complexity check.

In [ ]:
# Initialize model
model = ResNetCIFAR().to(device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,} ({total_params/1e6:.3f}M)")

Total params: 272,474 (0.272M)


**3. Baseline Model Training**

In [10]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Choose optimizer ("sgd" or "adam")
optimizer_type = "sgd"

if optimizer_type == "sgd":
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
elif optimizer_type == "adam":
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

epochs = 3

print("Starting training...")

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        # Metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100.0 * correct / total

    # ---- Evaluation ----
    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()

    test_acc = 100.0 * test_correct / test_total

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Loss: {running_loss/len(trainloader):.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}%"
    )

print("Finished training!")

Starting training...
Epoch [1/3] | Loss: 1.5367 | Train Acc: 42.50% | Test Acc: 45.81%
Epoch [2/3] | Loss: 1.0027 | Train Acc: 64.24% | Test Acc: 62.05%
Epoch [3/3] | Loss: 0.7874 | Train Acc: 72.27% | Test Acc: 66.83%
Finished training!


This experiment shows the baseline training performance of the ResNet-based model on the CIFAR-10 dataset over three epochs. The model was trained using cross-entropy loss, and performance was evaluated using both training and test accuracy after each epoch to measure learning progress and generalization.

Across training, the model shows steady improvement. The training loss decreases from 1.5367 to 0.7874, while training accuracy increases from 42.50% to 72.27%, indicating that the model is effectively learning discriminative features from the dataset. Similarly, test accuracy improves from 45.81% to 66.83%, showing consistent gains in generalization performance without obvious overfitting during this short training period. Overall, these results provide a strong baseline performance that will be used for comparison against sparse training methods in later experiments.